# AI/ML Assignment: Academic Study Assistant (RAG)

## Objective
This project builds a simple AI-powered study assistant using Retrieval Augmented Generation (RAG).  
It answers questions based on course materials (PDF notes).

## Subject Chosen
Software Engineering

## Approach
- Load PDF notes
- Split into chunks
- Store embeddings in vector database
- Retrieve relevant content
- Generate answers using LLM
- Used Ollama (local LLM) instead of OpenAI to avoid API cost and rate limits.
## Experiment 1: Chunking Strategies Comparison

In this experiment, we compare:
- Fixed-size chunking
- Sentence-based chunking

Goal: To check which gives better answers.

In [1]:
!pip install langchain langchain-community chromadb sentence-transformers langchain-huggingface


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.llms import Ollama

In [3]:
pdf_files = [
    "data/SE_NOTES_2.pdf",
    "data/SE_NOTES.pdf"
]

docs = []

for file in pdf_files:
    loader = PyPDFLoader(file)
    docs.extend(loader.load())

print("Total pages:", len(docs))

Total pages: 124


## Data Analysis

### Type of Documents
- PDF lecture notes

### Structure
- Headings, paragraphs, bullet points

### Challenges
1. Some formatting is inconsistent
2. Diagrams are not included in text
3. Some sections are not properly structured

In [4]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2"
)

In [5]:
# Fixed Chunking
splitter_fixed = CharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks_fixed = splitter_fixed.split_documents(docs)

db_fixed = Chroma.from_documents(chunks_fixed, embeddings)

print("Total Fixed chunks:", len(chunks_fixed))

Total Fixed chunks: 124


In [6]:
# Sentence Chunking
splitter_sentence = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks_sentence = splitter_sentence.split_documents(docs)

db_sentence = Chroma.from_documents(chunks_sentence, embeddings)
print("Total Sentence chunks:", len(chunks_sentence))

Total Sentence chunks: 615


In [7]:
llm = Ollama(model="mistral")

C:\Users\mratu\AppData\Local\Temp\ipykernel_12968\86901005.py:1: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  llm = Ollama(model="mistral")


## Test Questions

1. What is Software Engineering?
2. What is SDLC?
3. Explain Agile model
4. What is Waterfall model?
5. Difference between Agile and Waterfall
6. What is testing?
7. What is requirement analysis?
8. What is design phase?
9. What is maintenance phase?
10. What is debugging?

In [8]:
questions = [
    "What is Software Engineering?",
    "What is SDLC?",
    "Explain Agile model",
    "What is Waterfall model?",
    "Difference between Agile and Waterfall",
    "What is software testing?",
    "What is requirement analysis?",
    "What is design phase?",
    "What is maintenance phase?",
    "What is debugging?"
]

for q in questions:

    print("\n====================================")
    print("Question:", q)

    # FIXED CHUNKING
    print("\n----- FIXED CHUNKING -----")
    results = db_fixed.similarity_search(q, k=3)
    context = " ".join([r.page_content for r in results])

    response = llm.invoke(f"""
    Answer clearly and briefly based only on the context below:
    
    Context:
    {context}
    
    Question: {q}
    """)

    print("A:", response)

    # SENTENCE CHUNKING
    print("\n----- SENTENCE CHUNKING -----")
    results = db_sentence.similarity_search(q, k=3)
    context = " ".join([r.page_content for r in results])

    response = llm.invoke(f"""
    Answer clearly and briefly based only on the context below:
    
    Context:
    {context}
    
    Question: {q}
    """)

    print("A:", response)


Question: What is Software Engineering?

----- FIXED CHUNKING -----
A:  Software Engineering is the systematic approach to developing and maintaining software products efficiently and reliably, utilizing well-defined scientific principles, methods, and tools. It involves more than just program code, also encompassing associated libraries and documentation. A software product is developed for a specific requirement.

----- SENTENCE CHUNKING -----
A:  Software Engineering is a systematic approach to develop, maintain, and produce efficient and reliable software products. It involves the application of theories, methods, and tools within the field of engineering for the development of software.

Question: What is SDLC?

----- FIXED CHUNKING -----
A:  SDLC (Software Development Life Cycle) is not explicitly mentioned in the provided context. However, it can be inferred that SDLC refers to a process used by the software engineering team to develop software. This process likely includes ste

## Analysis

Sentence chunking performs better as it preserves complete sentences and context.
Fixed chunking sometimes splits important information.
Therefore, sentence-based chunking is more effective for this dataset.

## Results Summary

| Question Type | Fixed Chunking | Sentence Chunking |
|--------------|--------------|------------------|
| Definitions | Good | Good |
| Conceptual | Good | Better |
| Detailed Explanation | Average | Excellent |
| Overall | Good | Better |

In [9]:
def basic_prompt(context, question):
    return f"""
    Answer based on the context below:

    {context}

    Question: {question}
    """

In [10]:
def improved_prompt(context, question):
    return f"""
    You are a helpful academic assistant.

    Answer clearly with explanation and examples if possible.
    Keep the answer structured and easy to understand.

    Context:
    {context}

    Question: {question}
    """

In [12]:
for q in questions:

    print("\n====================================")
    print("Question:", q)

    results = db_sentence.similarity_search(q, k=3)
    context = " ".join([r.page_content for r in results])

    # BASIC PROMPT
    print("\n----- BASIC PROMPT -----")
    response_basic = llm.invoke(basic_prompt(context, q))
    print("A:", response_basic)

    # IMPROVED PROMPT
    print("\n----- IMPROVED PROMPT -----")
    response_improved = llm.invoke(improved_prompt(context, q))
    print("A:", response_improved)


Question: What is Software Engineering?

----- BASIC PROMPT -----
A:  Software Engineering refers to the application of scientific principles, methods, and tools to develop professional software in a systematic, cost-effective, and efficient manner. It involves creating a software product for a specific requirement or need, which is considered to be more than just program code, including executable programming, associated libraries, and documentation. Unlike traditional manufacturing, software does not wear out, but instead, software engineering aims to produce an efficient and reliable software product.

----- IMPROVED PROMPT -----
A:  Software Engineering refers to the application of engineering principles and methods to the development, maintenance, and improvement of computer software. It involves creating a software product that meets specific requirements in an efficient, reliable, and cost-effective manner.

Software Engineering differs from manufacturing in that software is no

## Prompt Comparison

### Basic Prompt
Answer based on the context below.

### Improved Prompt
- Clear explanation
- Structured output
- Academic style response

## Results Summary

| Question Type | Basic Prompt | Improved Prompt |
|--------------|------------|----------------|
| Definitions | Good | Better |
| Conceptual | Average | Better |
| Detailed | Average | Much Better |

## Analysis

- Improved prompt provides more structured and detailed answers.
- Basic prompt gives shorter and less explanatory responses.
- Improved prompt helps the model understand that explanation is required.
- For academic content, structured answers improve clarity and understanding.

Conclusion:
Improved prompting strategy performs better for this domain.